# M3. LLM 퓨샷 NER과 비교 — Claude에게 공식 태그 정의만 주고 시키면?

**질문**: 파인튜닝한 NER 모델 없이 LLM에 태그 정의와 예시 몇 개만 주면 얼마나 맞히는가?
이 숫자는 두 군데에 쓰인다.
1. "LLM이 있는데 왜 NER 모델을 학습하나"에 대한 답 (정확도·속도·비용 비교).
2. 2단계에서 **LLM으로 금융 라벨을 붙이는 방식**을 믿어도 되는지의 사전 근거. 기존 7종을 LLM이 얼마나 맞히는지가 금융 라벨 품질의 기대치가 된다.

**설계**
- 대상: test에서 무작위 200문장 (파인튜닝 모델과 **같은 200문장**으로 비교).
- 입력: 공식 태그 정의·라벨링 규칙(저작도구 매뉴얼) + train에서 뽑은 예시 8문장 + 문장.
- 출력: JSON으로 개체 텍스트와 태그만 받는다. **위치는 코드가 찾는다**(데이터 전처리와 같은 규칙: 모든 출현, 겹치면 긴 것 우선). LLM은 글자 위치를 세는 데 약하기 때문이다.
- 채점: 다른 실험과 같은 엄격 일치 span F1.

## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

In [1]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive
GPU: 없음


## 1. API 키와 모델·비용
키는 **Colab 왼쪽 열쇠 아이콘(보안 비밀)** 에 `ANTHROPIC_API_KEY` 라는 이름으로 넣는다. 노트북 셀, 파일, 깃허브에 키를 직접 쓰지 않는다. 비용은 그 키 주인의 계정에서 나간다.

**모델**: 비용을 낮추려고 **Claude Sonnet 5.5** (`claude-sonnet-5-5`)를 기본으로 쓴다. 가격(100만 토큰당, 2026-10 기준): 입력 $2 · 출력 $10 · 캐시 읽기 $0.20. (참고: Opus 5.5는 $4·$20, Haiku 5.5는 $0.10·$0.50 — 더 싸지만 정확도는 재 봐야 한다.)

**먼저 20문장으로 시험**: `N_EVAL = 20` 으로 돌려 아래 실행 셀이 출력하는 **실제 비용**을 보고, 괜찮으면 `N_EVAL = 200` 으로 바꿔 다시 실행한다. 끝난 문장은 건너뛰므로 앞의 20문장 비용이 다시 들지 않는다(같은 200문장의 앞 20개).

**예상 비용(추정, 실측 아님)**: 문장당 입력 약 2천 토큰(정의+예시, 캐시가 걸리면 두 번째부터 10% 가격) + 출력 수백 토큰(생각 포함). Sonnet 5.5 기준 20문장 약 0.1~0.2달러, 200문장 약 1~2달러.


In [2]:
!pip -q install -U anthropic
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
import anthropic, random
import ner_lib
client = anthropic.Anthropic()
MODEL = "claude-opus-5-5"     # 비용 절감. 바꿀 수 있는 값: "claude-opus-5-5", "claude-haiku-5-5"
N_EVAL = 100                     # 먼저 20으로 비용 확인 → 괜찮으면 200
N_POOL = 200                    # 비교 대상 200문장(고정). N_EVAL은 이 중 앞에서부터
PRICE = {"claude-sonnet-5-5": (2.0, 10.0), "claude-opus-5-5": (4.0, 20.0), "claude-haiku-5-5": (0.10, 0.50)}  # 입력·출력 $/100만 토큰


## 2. 프롬프트: 공식 정의 + 규칙 + 예시
시스템 프롬프트는 모든 문장에서 똑같이 쓰므로 **프롬프트 캐시**를 걸어 두 번째 호출부터 싸게 만든다. 예시는 train에서 개체가 있는 문장 8개를 고정 시드로 뽑는다(test 문장은 절대 예시로 쓰지 않는다).

In [3]:
GUIDE = '''너는 한국어 뉴스 문장에서 개체명을 찾는 라벨러다. 아래 태그 정의와 규칙을 지켜 개체 텍스트와 태그를 낸다.

태그 7종 (AI Hub 웹데이터 말뭉치 공식 정의)
- OG (ORGANIZATION): 기관 및 조직. 기관/단체, 회의/회담 명칭
- AF (ARTIFACTS): 인공물. 사람에 의해 창조된 대상물
- CV (CIVILIZATION): 문화 용어. 문명/문화와 관련된 용어 (직업·직위도 여기 포함)
- FD (STUDY_FIELD): 학문 분야. 과학, 사회과학, 의학, 예술, 철학 관련 학파 및 유파
- TR (THEORY): 이론, 법칙, 원리 등의 명칭
- TM (TERM): 위 정의 이외의 세분류에 해당하는 개체명 용어
- NW (NEW_WORD): 표준국어대사전과 우리말샘에 없는 신조어
인물·장소·날짜·수량은 태그가 없으므로 내지 않는다. 사람 이름은 (이름)으로 가려져 있다.

규칙
- 개체 텍스트는 문장에 나온 글자 그대로 쓴다. 조사·어미는 빼고 개체까지만 쓴다. 예) 학생들 → 학생
- 여러 어절이 하나의 대상을 가리키면 전체를 하나로: 현대 자동차 → 현대 자동차(OG)
- 나누어도 뜻이 유지되면 나눈다: 척추뼈 디스크 → 척추뼈(TM), 디스크(TM)
- 괄호 앞뒤를 합쳐야 완전하면 괄호까지 포함: 진(Jin)에어(OG). 괄호 안이 줄임말·원어면 각각: 방송통신위원회(OG), 방통위(OG)
- 개체가 없으면 빈 목록을 낸다.'''

train_rows = load_jsonl(f"{SPLIT}/train.jsonl")
rnd = random.Random(7)
shots = rnd.sample([r for r in train_rows if 2 <= len(r["spans"]) <= 6 and len(r["text"]) < 120], 8)
def as_answer(r):
    return json.dumps({"entities": [{"text": r["text"][s:e], "tag": t} for s, e, t in r["spans"]]}, ensure_ascii=False)
EXAMPLES = "\n\n".join(f"문장: {r['text']}\n답: {as_answer(r)}" for r in shots)
SYSTEM = [{"type": "text", "text": GUIDE + "\n\n예시\n\n" + EXAMPLES, "cache_control": {"type": "ephemeral"}}]
print(SYSTEM[0]["text"][-600:])

ext": "자유한국당", "tag": "OG"}, {"text": "초선의원", "tag": "CV"}]}

문장: 인천시교육청, 학부모로 구성된 시민방역단 출범
답: {"entities": [{"text": "인천시교육청", "tag": "OG"}, {"text": "학부모", "tag": "CV"}]}

문장: 느껴진다' 등 반응을 남겼다.회사 내에서 조 회장은 임직원과의 소통에 적극 나서온 것으로 평가받는다.
답: {"entities": [{"text": "회장", "tag": "CV"}, {"text": "임직원", "tag": "CV"}]}

문장: 당초 일동제약은 '라스미디탄'을 미국 제약기업 콜루시드로부터 사들인 것이지만 릴리가 올초 콜루시드를 9억6000만달러에 인수했다.
답: {"entities": [{"text": "일동제약", "tag": "OG"}, {"text": "라스미디탄", "tag": "TM"}]}

문장: ▶ 경복궁과 가깝고 한옥마을인 북촌 초입인 데다 여중·고 4곳이 몰려있는 곳이다.
답: {"entities": [{"text": "경복궁", "tag": "AF"}, {"text": "한옥마을", "tag": "CV"}]}


## 3. 호출 함수
- **구조화 출력**(`output_config.format` 의 JSON 스키마)으로 답 형식을 고정한다. 파싱 실패가 없다.
- `effort: "low"`: 문장 하나에서 개체를 고르는 비교적 단순한 일이라 생각을 짧게 하게 해 비용과 시간을 줄인다.
- **거절 대비**: 드물게 안전 분류기가 요청을 거절할 수 있다. `fallbacks: "default"` 를 켜 두면 서버가 다른 모델로 자동 재시도한다. 그래도 거절이면 그 문장은 빈 답으로 기록하고 개수를 센다.
- 위치 찾기: LLM이 낸 (텍스트, 태그)를 `prepare_data.make_spans` 와 같은 규칙으로 span으로 바꾼다. 문장에 없는 텍스트는 버리고 개수를 센다(환각 지표).

In [4]:
from prepare_data import make_spans
SCHEMA = {"type": "object", "additionalProperties": False, "required": ["entities"],
          "properties": {"entities": {"type": "array", "items": {
              "type": "object", "additionalProperties": False, "required": ["text", "tag"],
              "properties": {"text": {"type": "string"},
                             "tag": {"type": "string", "enum": ["CV", "OG", "TM", "AF", "FD", "TR", "NW"]}}}}}}

def llm_ner(text):
    resp = client.beta.messages.create(
        model=MODEL, max_tokens=4000, system=SYSTEM,
        betas=["server-side-fallback-2026-07-01"], fallbacks="default",
        output_config={"effort": "low", "format": {"type": "json_schema", "schema": SCHEMA}},
        messages=[{"role": "user", "content": f"문장: {text}"}])
    if resp.stop_reason == "refusal":
        return [], resp.usage, "refusal"
    txt = [b.text for b in resp.content if b.type == "text"][-1]
    ents = json.loads(txt)["entities"]
    return ents, resp.usage, "ok"

In [5]:
import time
test_rows = load_jsonl(f"{SPLIT}/test.jsonl")
sample = random.Random(42).sample(test_rows, N_POOL)[:N_EVAL]     # 항상 같은 200문장의 앞 N_EVAL개
out_path = f"{RESULTS}/llm_raw/M3_llm_{MODEL}_raw{N_POOL}.jsonl"   # LLM 원답(채점용 예측 파일과 따로 둠)
os.makedirs(os.path.dirname(out_path), exist_ok=True)
done = {json.loads(l)["id"] for l in open(out_path, encoding="utf-8")} if os.path.exists(out_path) else set()
usage = {"input": 0, "cache_read": 0, "cache_write": 0, "output": 0}; status = {"ok": 0, "refusal": 0, "error": 0}
t0 = time.time()
with open(out_path, "a", encoding="utf-8") as f:
    for i, r in enumerate(sample):
        if r["id"] in done: continue
        try:
            ents, u, st = llm_ner(r["text"])
        except anthropic.RateLimitError:
            time.sleep(30); ents, u, st = llm_ner(r["text"])
        except anthropic.APIStatusError as e:
            print("API 오류", e.status_code, r["id"]); status["error"] += 1; continue
        status[st] += 1
        usage["input"] += u.input_tokens; usage["output"] += u.output_tokens
        usage["cache_read"] += u.cache_read_input_tokens or 0; usage["cache_write"] += u.cache_creation_input_tokens or 0
        labs = [{"text": e["text"], "tag": e["tag"]} for e in ents]
        spans = make_spans(r["text"], labs)
        missing = sum(1 for e in ents if e["text"] not in r["text"])
        f.write(json.dumps({"id": r["id"], "category": r["category"], "text": r["text"], "gold": r["spans"],
                            "pred": spans, "llm_raw": ents, "not_in_text": missing, "status": st}, ensure_ascii=False) + "\n")
        if (i + 1) % 20 == 0: print(i + 1, "문장", round(time.time() - t0), "초")
pin, pout = PRICE[MODEL]
cost = (usage["input"] * pin + usage["cache_write"] * pin * 1.25 + usage["cache_read"] * pin * 0.1 + usage["output"] * pout) / 1e6
print(status, usage, f"이번 실행 비용 약 ${cost:.2f} (캐시 쓰기 1.25배·읽기 0.1배 가정)")

20 문장 47 초
40 문장 86 초
60 문장 132 초
80 문장 175 초
100 문장 217 초
{'ok': 100, 'refusal': 0, 'error': 0} {'input': 8821, 'cache_read': 190674, 'cache_write': 1926, 'output': 8098} 이번 실행 비용 약 $0.28 (캐시 쓰기 1.25배·읽기 0.1배 가정)


## 4. 채점과 비교
같은 200문장에 대해 파인튜닝 모델(M1 결과의 예측 파일)과 나란히 본다. LLM이 문장에 없는 텍스트를 낸 횟수(`not_in_text`)는 환각 지표로 따로 적는다.

In [6]:
import evaluate as ev
llm = [json.loads(l) for l in open(out_path, encoding="utf-8")]
ids = {r["id"] for r in llm}
gold_sub = [r for r in test_rows if r["id"] in ids]                      # 같은 200문장
llm_pred = {r["id"]: r["pred"] for r in llm}
name = f"M3_llm_{MODEL}"
ner_lib.save_preds(f"{RESULTS}/preds/{name}_test200.jsonl", gold_sub, [llm_pred[r["id"]] for r in gold_sub])
res = ev.score(gold_sub, ev.load_pred(f"{RESULTS}/preds/{name}_test200.jsonl"), name, "test200", RESULTS,
               {"member": MEMBER, "stage": "M3_LLM"})
m = res["metrics"]
print(f"LLM  F1 {m['ALL']['f1']}  P {m['ALL']['precision']}  R {m['ALL']['recall']}  macro {m['MACRO_F1']}"
      f" | 문장에 없는 개체 {sum(r['not_in_text'] for r in llm)}개 | 오답 {res['error_counts']}")
for r in load_results(RESULTS):                                           # 파인튜닝 모델을 같은 200문장으로
    p = f"{RESULTS}/preds/{r['exp']}_test.jsonl"
    if r["exp"].startswith("M1_") and os.path.exists(p):
        mm = ev.score(gold_sub, ev.load_pred(p), r["exp"], "test200", RESULTS)["metrics"]
        print(f"{r['exp']:24s} F1 {mm['ALL']['f1']}  P {mm['ALL']['precision']}  R {mm['ALL']['recall']}  macro {mm['MACRO_F1']}")
json.dump({"name": name, "model": MODEL, "member": MEMBER, "n_test": len(llm), "test200": m, "usage": usage,
           "not_in_text": sum(r["not_in_text"] for r in llm)},
          open(f"{RESULTS}/scores/M3_llm_summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

LLM  F1 0.3097  P 0.2109  R 0.5822  macro 0.1541 | 문장에 없는 개체 0개 | 오답 {'경계': 19, '유형': 10, '경계+유형': 17, '오탐': 272, '미탐': 12}
M1_aihub_electra_1       F1 0.6621  P 0.6667  R 0.6575  macro 0.3967
M1_aihub_electra_1_morph F1 0.6899  P 0.7021  R 0.6781  macro 0.4209
M1_aihub_electra_2       F1 0.676  P 0.6879  R 0.6644  macro 0.4122
M1_aihub_electra_2_200k  F1 0.7143  P 0.7095  R 0.7192  macro 0.4402
M1_aihub_electra_2_300k  F1 0.7379  P 0.7431  R 0.7329  macro 0.4661
M1_hong_klue-roberta-large_lr2e-5_300k F1 0.7148  P 0.7172  R 0.7123  macro 0.4217
M1_klue_bert             F1 0.6644  P 0.6577  R 0.6712  macro 0.4029
M1_klue_roberta          F1 0.6804  P 0.6828  R 0.6781  macro 0.4192
M1_klue_roberta_morph    F1 0.692  P 0.6993  R 0.6849  macro 0.3981
M1_koelectra             F1 0.6309  P 0.6184  R 0.6438  macro 0.3688
M1_koelectra_300k        F1 0.7372  P 0.7347  R 0.7397  macro 0.4583


In [7]:
# LLM과 정답이 갈린 예시 보기 — LLM이 맞고 라벨이 틀린 경우(라벨 노이즈)도 여기서 보인다
for r in random.Random(0).sample(llm, 5):
    g = {tuple(x) for x in r["gold"]}; p = {tuple(x) for x in r["pred"]}; t = r["text"]
    print("\n", t[:150]); print("  정답에만:", [(t[s:e], k) for s, e, k in sorted(g - p)]); print("  LLM에만:", [(t[s:e], k) for s, e, k in sorted(p - g)])


 【전주=뉴시스】(이름) 기자 = 전북대학교는 (이름) 대학원생(생리활성소재과학과 박사과정·지도교수 (이름))이 교육부와 한국연구재단이 지원하는 '2019년도 글로벌 박사 펠로우십'에 선정됐다고 23일 밝혔다.'글로벌 박사 펠로우십'은 국내 박사 및 석·박사 통합과정 및
  정답에만: [('박사', 'CV'), ('박사 펠로우십', 'CV'), ('박사 펠로우십', 'CV'), ('박사', 'CV')]
  LLM에만: [('뉴시스', 'OG'), ('생리활성소재과학과', 'OG'), ('박사과정', 'CV'), ('지도교수', 'CV'), ('글로벌 박사 펠로우십', 'CV'), ('글로벌 박사 펠로우십', 'CV'), ('석·박사 통합과정', 'CV')]

 또한, 존박은 드라마 '동백꽃 필 무렵'의 ost '(이름) 사람'으로 큰 사랑을 받기도 했다..
  정답에만: []
  LLM에만: [('드라마', 'CV'), ('동백꽃 필 무렵', 'AF'), ('ost', 'CV')]

 기획재정부가 20일 발간한 '월간 재정동향 5월호'에 따르면 올해 3월까지 사회보장성 기금수지를 제외한 관리재(이름) 누계는 -24조8000억원이다.
  정답에만: []
  LLM에만: [('기획재정부', 'OG'), ('월간 재정동향 5월호', 'AF'), ('사회보장성 기금수지', 'TM')]

 시진의 강모연 구출 작전은 성공할 수 있을 것인가.
  정답에만: []
  LLM에만: []

 [청주=뉴시스] (이름) 기자 = 신종 코로나바이러스 감염증(코로나19) 확산으로 어려움에 처한 충북지역에 연일 각계의 온정이 답지하고 있다.
  정답에만: [('코로나바이러스 감염증', 'TM')]
  LLM에만: [('뉴시스', 'OG'), ('신종 코로나바이러스 감염증', 'TM'), ('코로나19', 'TM')]
